## Creating streaming read on Table

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, current_timestamp


# ============================================================
# SILVER → GOLD : PRODUCTS SCD TYPE 2
# ============================================================

# 1. Prepare the source data
@dp.materialized_view(
    name="products_source"
)
def products_source():

    return (
        spark.read.table("databricks_cata.silver.products_silver")
    )

dp.create_streaming_table(
    name="DimProducts"
)
# 2. Apply CDC and maintain SCD Type 2 history
dp.create_auto_cdc_flow(
    target="DimProducts",
    source="products_source",

    # Business/natural key
    keys=["product_id"],

    # Determines the order of changes
    sequence_by=col("product_id"),

    # SCD Type 2
    stored_as_scd_type=2
)
